### Library

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
import multiprocessing as mp
if mp.get_start_method(allow_none=True) is None:
    mp.set_start_method('spawn')

import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', message='.*input_shape.*')
warnings.filterwarnings('ignore', message='.*structure of.*inputs.*')

import os, time, gc
from types import SimpleNamespace

import numpy as np
import pandas as pd
from scipy.special import gamma

import jax, jax.numpy as jnp

import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber
from tensorflow.keras import backend as K

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder

import optuna
import plotly.io as pio

### Environment Setting

In [ ]:
np_f32 = np.float32
jnp_f32 = jnp.float32
dtype_basis = np.float32

jax.config.update("jax_enable_x64", False)

pio.renderers.default = "notebook"
warnings.filterwarnings("ignore", category=UserWarning)

os.environ.update({"TF_CPP_MIN_LOG_LEVEL": "2"})
optuna.logging.set_verbosity(optuna.logging.WARNING)

os.environ.setdefault("OMP_NUM_THREADS", "12")
os.environ.setdefault("MKL_NUM_THREADS", "12")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "12")

def init_hardware(dtype: str = "float32"):
    gpus = tf.config.list_physical_devices("GPU")
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    strategy = (tf.distribute.MirroredStrategy() if len(gpus) > 1 else tf.distribute.get_strategy())
    mixed_precision.set_global_policy(dtype)
    return strategy

strategy = init_hardware(dtype="float32")

### Our Functions

In [ ]:
from spherical_deepkriging.basis_functions.wendland.wenland import wendland
from spherical_deepkriging.models.deep_kriging import DeepKrigingTrainer
from spherical_deepkriging.models.universal_kriging import UniversalKriging
from spherical_deepkriging.basis_functions.mrts.mrts import mrts0

from rpy2.robjects.conversion import localconverter
from spherical_deepkriging.basis_functions.mrts_sphere.sphere import mrts_sphere, numpy2ri_converter

### Helper

In [ ]:
def data_preprocessing(data_path, variable, num_sample, seed):
    data = (pd.read_csv(data_path).sample(n=num_sample, random_state=seed).replace("-", np.nan).dropna())

    numeric_cols = ["longitude", "latitude", variable]
    data[numeric_cols] = data[numeric_cols].apply(pd.to_numeric, errors="coerce")
    data.dropna(subset=numeric_cols, inplace=True)

    lon, lat = data["longitude"].to_numpy(), data["latitude"].to_numpy()

    norm_lon = (lon - lon.min()) / (lon.max() - lon.min())
    norm_lat = (lat - lat.min()) / (lat.max() - lat.min())

    location_data = np.column_stack([lat, lon]).astype("float32")
    location_data_norm = np.column_stack([norm_lon, norm_lat]).astype("float32")
    y_combined = data[variable].to_numpy().astype("float32")[:, None]

    # Handle
    categorical_data = None

    return location_data, location_data_norm, categorical_data, y_combined


def precompute_wendland(location, num_basis):
    parts = []
    for nb in num_basis:
        grid = np.column_stack(np.meshgrid(
            np.linspace(0, 1, int(np.sqrt(nb)), dtype=np_f32),
            np.linspace(0, 1, int(np.sqrt(nb)), dtype=np_f32),
        )).reshape(-1, 2).astype(np_f32)

        theta = np_f32(2.5)/np_f32(np.sqrt(nb))
        parts.append(
            wendland(location, grid, theta=theta, k=2)
        )

    return np.hstack(parts).astype(dtype_basis, copy=False)


def precompute_max_mrts(distance_type, location_data, knot_num, order_max, knot=None):
    if knot is None:
        idx_knot = np.random.choice(location_data.shape[0], knot_num, replace=False)
        knot = location_data[idx_knot].astype(np_f32)
    else:
        idx_knot = None

    if distance_type == "sphere":
        with localconverter(numpy2ri_converter):
            res_r = mrts_sphere(knot, order_max, location_data.astype(np_f32))
        res_dict = dict(zip(res_r.names(), res_r))
        phi = np.asarray(res_dict["mrts"], dtype=dtype_basis)
    else:
        phi = np.asarray(
            mrts0(jnp.asarray(knot, dtype=jnp_f32), k=order_max, 
                  x=jnp.asarray(location_data, dtype=jnp_f32)), dtype=dtype_basis
        )

    return phi, idx_knot, knot


def prepare_data(categorical_data, basis, y_combined, seed, split_ratio=(0.8, 0.1, 0.1)):
    idx_all = np.arange(basis.shape[0])
    train_ratio, val_ratio, test_ratio = split_ratio
    
    train_val_x1, test_x1 = train_test_split(
        idx_all, train_size=train_ratio+val_ratio, random_state=seed)
    train_x1, val_x1 = train_test_split(
        train_val_x1, train_size=train_ratio/(train_ratio+val_ratio), random_state=seed)
    
    X_train_cont, X_val_cont, X_test_cont = (
        basis[train_x1], basis[val_x1], basis[test_x1])
    y_train, y_val, y_test = (
        y_combined[train_x1], y_combined[val_x1], y_combined[test_x1])
    
    def flatten(targets):
        return targets.reshape(-1).astype(np_f32, copy=False)
    y_train_flat, y_val_flat, y_test_flat = map(flatten, [y_train, y_val, y_test])

    def flatten(covariates):
        return covariates.reshape(-1, basis.shape[1]).astype(np_f32)
    X_train_cont_flat, X_val_cont_flat, X_test_cont_flat = map(flatten, [X_train_cont, X_val_cont, X_test_cont])
    
    # Handle categorical features
    if categorical_data is None:
        zero_vector = lambda n: np.zeros((n, 0), dtype=np_f32)
        X_train_cat, X_val_cat, X_test_cat = map(zero_vector, [len(X_train_cont_flat), len(X_val_cont_flat), len(X_test_cont_flat)])
    else:
        cat_train = categorical_data.reshape(-1, 1)[train_x1]
        cat_val = categorical_data.reshape(-1, 1)[val_x1]
        cat_test = categorical_data.reshape(-1, 1)[test_x1]
        
        OHE = OneHotEncoder(sparse_output=False, dtype=np_f32)
        X_train_cat = OHE.fit_transform(cat_train).astype(np_f32)
        X_val_cat = OHE.transform(cat_val).astype(np_f32)
        X_test_cat = OHE.transform(cat_test).astype(np_f32)
    
    return (X_train_cont_flat, X_train_cat, y_train_flat,
            X_val_cont_flat, X_val_cat, y_val_flat,
            X_test_cont_flat, X_test_cat, y_test_flat)


def create_config(input_dim, categorical_dim, loss, epochs,
                  hidden_layers, activation, dropout_rate, optimizer, batch_size, patience):
    return SimpleNamespace(
        input_dim=input_dim,
        num_categorical_features=categorical_dim,
        hidden_layers=hidden_layers,
        activation=activation,
        dropout_rate=dropout_rate,
        output_type='continuous',
        optimizer=optimizer,
        loss=loss,
        metrics=['mae'],
        epochs=epochs,
        batch_size=batch_size,
        patience=patience,
        verbose=0
    )


def create_config_original(input_dim, categorical_dim, loss, epochs, batch_size):
    return SimpleNamespace(
        input_dim=input_dim,
        num_categorical_features=categorical_dim,
        hidden_layers=[100, 100, 100],
        activation='elu',
        dropout_rate=0.0,
        output_type='continuous',
        optimizer=Adam(learning_rate=1e-3),
        loss=loss,
        metrics=['mse', 'mae'],
        epochs=epochs,
        batch_size=batch_size,
        verbose=0
    )


def train_eval(name_model, epochs, batch_size, loss,
               X_train, X_train_cat, y_train,
               X_val, X_val_cat, y_val,
               X_test, X_test_cat, y_test):

    if name_model in ["OLS_wendland", "OLS_sphere"]:
        t0 = time.time()
        model = LinearRegression().fit(X_train, y_train)

        val_loss = float(mean_squared_error(y_val, model.predict(X_val)))
        y_pred = model.predict(X_test).astype(np_f32).reshape(-1)
        training_time = time.time() - t0
        
        metrics = {
            "Model": name_model,
            "Val_loss": float(val_loss),
            "MSPE": float(mean_squared_error(y_test, y_pred)),
            "RMSE": float(np.sqrt(mean_squared_error(y_test, y_pred))),
            "MAE": float(mean_absolute_error(y_test, y_pred)),
            "R2": float(r2_score(y_test, y_pred)),
            "Time": float(training_time),
        }
        del model
        gc.collect()
        return metrics
    
    elif name_model == "DeepKriging_wendland":
        config = create_config_original(
            input_dim=X_train.shape[1],
            categorical_dim=X_train_cat.shape[1],
            loss=loss,
            epochs=epochs,
            batch_size=batch_size
        )

    elif name_model in ["DeepKriging_sphere", "DeepKriging_sphere_Huber"]:
        optimizer = Adam(learning_rate=5e-3)
        config = create_config(
            input_dim=X_train.shape[1],
            categorical_dim=X_train_cat.shape[1],
            loss=loss,
            epochs=epochs,
            hidden_layers=[1024, 512, 256, 128, 64],
            activation='relu',
            dropout_rate=0.3,
            optimizer=optimizer,
            batch_size=batch_size,
            patience=40
        )

    t0 = time.time()
    with strategy.scope():
        model = DeepKrigingTrainer(config)
        model.model.compile(optimizer=config.optimizer, loss=config.loss, metrics=config.metrics)

    checkpoint_path = f"best_{name_model}_{time.time_ns()}.weights.h5"
    if name_model == "DeepKriging_wendland":
        callbacks = [
            tf.keras.callbacks.ModelCheckpoint(
                filepath=checkpoint_path, monitor="val_loss", mode="min",
                save_best_only=True, save_weights_only=True, verbose=0)
        ]
    else:
        callbacks = [
            tf.keras.callbacks.ModelCheckpoint(
                filepath=checkpoint_path, monitor="val_loss", mode="min",
                save_best_only=True, save_weights_only=True, verbose=0),
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=config.patience, restore_best_weights=True, verbose=0),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=max(5, config.patience // 2), min_lr=1e-6, verbose=0)
        ]

    train_dataset = tf.data.Dataset.from_tensor_slices((
        (X_train, X_train_cat), y_train
    )).batch(config.batch_size).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((
        (X_val, X_val_cat), y_val
    )).batch(config.batch_size).prefetch(tf.data.AUTOTUNE)

    history = model.model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=epochs,
        callbacks=callbacks,
        verbose=0
    )

    if os.path.exists(checkpoint_path):
        model.model.load_weights(checkpoint_path)
        os.remove(checkpoint_path)
    
    val_loss = float(np.min(history.history["val_loss"]))
    y_pred = model.model.predict([X_test, X_test_cat], verbose=0).reshape(-1).astype(np_f32)
    training_time = time.time() - t0

    metrics = {
        "Model": name_model,
        "Val_loss": float(val_loss),
        "MSPE": float(mean_squared_error(y_test, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_test, y_pred))),
        "MAE": float(mean_absolute_error(y_test, y_pred)),
        "R2": float(r2_score(y_test, y_pred)),
        "Time": float(training_time),
    }

    del train_dataset, val_dataset, model
    K.clear_session()
    gc.collect()
    
    return metrics

### Experiment Setup

In [ ]:
# Model Setup
seed = 1234
epochs = 500
batch_size = 2048
num_sample = 200000
huber_delta = 1.345

# Data
variable = "precipitation"
data_path = "../../../data/precipitation_data/data_s_p_date01_hour01.csv"

# Basis Setup
# Wendland
num_basis = [10**2, 19**2, 37**2]

knot_num = 2000
order_max = 1400
base_orders = [50, 100, 200, 500, 700, 1000, 1400]

### Outcome

In [ ]:
# Store for results
Record = {}

# Data Preprocessing
location_data, location_data_norm, categorical_data, y_combined = data_preprocessing(data_path, variable, num_sample, seed)

# Compute Basis Functions
max_Phi_sphere, idx_knot, knot = precompute_max_mrts("sphere", location_data, knot_num, order_max, knot=None)
max_Phi_sphere = max_Phi_sphere.astype(dtype_basis, copy=False)

Phi_wendland = precompute_wendland(location_data_norm, num_basis)

In [ ]:
# OLS_wendland
parts = prepare_data(categorical_data, Phi_wendland, y_combined, seed)
metrics = train_eval(
    "OLS_wendland", None, None, None, *parts
)
Record["OLS_wendland"] = { 
    "RMSE": metrics["RMSE"], "MAE": metrics["MAE"], "R2": metrics["R2"], "Time": metrics["Time"], "Param": "--"
}
del parts; gc.collect()


# DeepKriging_wendland
parts = prepare_data(categorical_data, Phi_wendland, y_combined, seed)
with strategy.scope():
    metrics = train_eval(
        "DeepKriging_wendland", epochs, batch_size, "mse", *parts
    )
Record["DeepKriging_wendland"] = {
    "RMSE": metrics["RMSE"], "MAE": metrics["MAE"], "R2": metrics["R2"], "Time": metrics["Time"], "Param": "--"
}
del parts; K.clear_session(); gc.collect()


# Tuning order parameter for OLS_sphere
Best_OLS_S = {'order': None, 'metrics': None}
Results_order_OLS_S = []

print("\nTuning order parameter for OLS_sphere")
for order in base_orders:
    Phi_sphere = max_Phi_sphere[:, :order].astype(np_f32)
    parts = prepare_data(categorical_data, Phi_sphere, y_combined, seed)
    metrics = train_eval(
        "OLS_sphere", None, None, None, *parts
    )
    
    Results_order_OLS_S.append({'order': order, 'val_loss': metrics["Val_loss"], 'rmse': metrics["RMSE"], 'mae': metrics["MAE"], 'r2': metrics["R2"]})
    
    if Best_OLS_S['metrics'] is None or metrics["Val_loss"] < Best_OLS_S['metrics']["Val_loss"]:
        Best_OLS_S['order'] = order
        Best_OLS_S['metrics'] = metrics
    
    del Phi_sphere, parts; gc.collect()

print(f"{'Order':<8} {'Val Loss':<10} {'RMSE':<10} {'MAE':<10} {'R²':<10}")
print("-"*48)
for res in Results_order_OLS_S:
    marker = " *" if res['order'] == Best_OLS_S['order'] else ""
    print(f"{res['order']:<8} {res['val_loss']:<10.4f} {res['rmse']:<10.4f} {res['mae']:<10.4f} {res['r2']:<10.4f}{marker}")

Record["OLS_sphere"] = {
    "RMSE": Best_OLS_S['metrics']["RMSE"], "MAE": Best_OLS_S['metrics']["MAE"], "R2": Best_OLS_S['metrics']["R2"], "Time": Best_OLS_S['metrics']["Time"], "Param": Best_OLS_S['order']
}


# Tuning order parameter for DeepKriging_sphere
Best_DK_S = {'order': None, 'metrics': None}
Results_order_DK_S = []

print("\nTuning order parameter for DeepKriging_sphere")
for order in base_orders:
    Phi_sphere = max_Phi_sphere[:, :order].astype(np_f32)
    parts = prepare_data(categorical_data, Phi_sphere, y_combined, seed)
    with strategy.scope():
        metrics = train_eval(
            "DeepKriging_sphere", epochs, batch_size, "mse", *parts
        )
    
    Results_order_DK_S.append({'order': order, 'val_loss': metrics["Val_loss"], 'rmse': metrics["RMSE"], 'mae': metrics["MAE"], 'r2': metrics["R2"]})
    
    if Best_DK_S['metrics'] is None or metrics["Val_loss"] < Best_DK_S['metrics']["Val_loss"]:
        Best_DK_S['order'] = order
        Best_DK_S['metrics'] = metrics
    
    del Phi_sphere, parts; K.clear_session(); gc.collect()

# Print results
print(f"{'Order':<8} {'Val Loss':<10} {'RMSE':<10} {'MAE':<10} {'R²':<10}")
print("-"*48)
for res in Results_order_DK_S:
    marker = " *" if res['order'] == Best_DK_S['order'] else ""
    print(f"{res['order']:<8} {res['val_loss']:<10.4f} {res['rmse']:<10.4f} {res['mae']:<10.4f} {res['r2']:<10.4f}{marker}")

Record["DeepKriging_sphere"] = {
    "RMSE": Best_DK_S['metrics']["RMSE"], "MAE": Best_DK_S['metrics']["MAE"], "R2": Best_DK_S['metrics']["R2"], "Time": Best_DK_S['metrics']["Time"], "Param": Best_DK_S['order']
}


# Tuning order parameter for DeepKriging_sphere_Huber
Best_DK_S_H = {'order': None, 'metrics': None}
Results_order_DK_S_H = []

print("\nTuning order parameter for DeepKriging_sphere_Huber")
for order in base_orders:
    Phi_sphere = max_Phi_sphere[:, :order].astype(np_f32)
    parts = prepare_data(categorical_data, Phi_sphere, y_combined, seed)
    with strategy.scope():
        metrics = train_eval(
            "DeepKriging_sphere_Huber", epochs, batch_size, Huber(delta=huber_delta), *parts
        )
    
    Results_order_DK_S_H.append({'order': order, 'val_loss': metrics["Val_loss"], 'rmse': metrics["RMSE"], 'mae': metrics["MAE"], 'r2': metrics["R2"]})
    
    if Best_DK_S_H['metrics'] is None or metrics["Val_loss"] < Best_DK_S_H['metrics']["Val_loss"]:
        Best_DK_S_H['order'] = order
        Best_DK_S_H['metrics'] = metrics
    
    del Phi_sphere, parts; K.clear_session(); gc.collect()

# Print results
print(f"{'Order':<8} {'Val Loss':<10} {'RMSE':<10} {'MAE':<10} {'R²':<10}")
print("-"*48)
for res in Results_order_DK_S_H:
    marker = " *" if res['order'] == Best_DK_S_H['order'] else ""
    print(f"{res['order']:<8} {res['val_loss']:<10.4f} {res['rmse']:<10.4f} {res['mae']:<10.4f} {res['r2']:<10.4f}{marker}")

Record["DeepKriging_sphere_Huber"] = {
    "RMSE": Best_DK_S_H['metrics']["RMSE"], "MAE": Best_DK_S_H['metrics']["MAE"], "R2": Best_DK_S_H['metrics']["R2"], "Time": Best_DK_S_H['metrics']["Time"], "Param": Best_DK_S_H['order']
}


# Tuning order parameter for UniversalKriging
Best_UK = {'order': None, 'metrics': None}
Results_UK = []

print("\nTuning order parameter for UniversalKriging")
for order in base_orders:
    Phi_sphere = max_Phi_sphere[:, :order].astype(np_f32)
    
    # Prepare data splits
    idx_all = np.arange(Phi_sphere.shape[0])
    train_val_idx, test_idx = train_test_split(idx_all, train_size=0.9, random_state=seed)
    train_idx, val_idx = train_test_split(train_val_idx, train_size=8/9, random_state=seed)
    
    # Prepare coordinates, basis and y
    coords_train, coords_val, coords_test = location_data[train_idx], location_data[val_idx], location_data[test_idx]
    phi_train, phi_val, phi_test = Phi_sphere[train_idx], Phi_sphere[val_idx], Phi_sphere[test_idx]
    y_train, y_val, y_test = y_combined[train_idx].flatten(), y_combined[val_idx].flatten(), y_combined[test_idx].flatten()
    
    # Train UniversalKriging
    t0 = time.time()
    uk_model = UniversalKriging(num_neighbors=30, cov_function='exponential')
    uk_model.fit(coords_train, phi_train, y_train, center_y=True)
    
    # Predict on validation set
    y_pred_val = uk_model.predict(coords_val, phi_val, return_centered=True)
    val_loss = mean_squared_error(y_val - uk_model.y_mean, y_pred_val)
    
    # Predict on test set
    y_pred_test = uk_model.predict(coords_test, phi_test, return_centered=False)
    training_time = time.time() - t0

    metrics = {
        "Val_loss": float(val_loss),
        "RMSE": float(np.sqrt(mean_squared_error(y_test, y_pred_test))),
        "MAE": float(mean_absolute_error(y_test, y_pred_test)),
        "R2": float(r2_score(y_test, y_pred_test)),
        "Time": training_time
    }
    
    Results_UK.append({
        'order': order,
        'val_loss': metrics["Val_loss"],
        'rmse': metrics["RMSE"],
        'mae': metrics["MAE"],
        'r2': metrics["R2"]
    })
    
    if Best_UK['metrics'] is None or metrics["Val_loss"] < Best_UK['metrics']["Val_loss"]:
        Best_UK['order'] = order
        Best_UK['metrics'] = metrics
    
    # Cleanup
    uk_model.cleanup(); del uk_model, Phi_sphere; gc.collect()

# Print results
print(f"{'Order':<8} {'Val Loss':<10} {'RMSE':<10} {'MAE':<10} {'R²':<10}")
print("-"*48)
for res in Results_UK:
    marker = " *" if res['order'] == Best_UK['order'] else ""
    print(f"{res['order']:<8} {res['val_loss']:<10.4f} {res['rmse']:<10.4f} {res['mae']:<10.4f} {res['r2']:<10.4f}{marker}")

Record["UniversalKriging"] = {"RMSE": Best_UK['metrics']["RMSE"], "MAE": Best_UK['metrics']["MAE"], "R2": Best_UK['metrics']["R2"], "Time": Best_UK['metrics']["Time"], "Param": Best_UK['order']}

In [ ]:
result_table = []
for model in ["OLS_wendland", "OLS_sphere", "DeepKriging_wendland", "DeepKriging_sphere", "DeepKriging_sphere_Huber", "UniversalKriging"]:
    result_table.append({
        "Model": model,
        "Param": Record[model]["Param"],
        "RMSE": f"{Record[model]['RMSE']:.4f}",
        "MAE": f"{Record[model]['MAE']:.4f}",
        "R²": f"{Record[model]['R2']:.4f}",
        "Times": f"{Record[model]['Time']:.4f}"
    })

df_results = pd.DataFrame(result_table)
print("\n", df_results.to_markdown(index=False, tablefmt="github"), sep="")

# Find best model
best_model = min(result_table, key=lambda x: float(x["RMSE"]))
print(f"\n🏆 Best Model: {best_model['Model']} (RMSE: {best_model['RMSE']})")